# Playground Series S6E4: Autoresearch Submission

This notebook is an **autoresearch submission** for the Kaggle competition **Playground Series - Season 6, Episode 4**. The goal is to predict `Irrigation_Need` as one of `Low`, `Medium`, or `High`, with balanced accuracy as the evaluation metric.

GitHub repository: [kaggle-s6e4-autoresearch](https://github.com/Andrew-Girgis/kaggle-s6e4-autoresearch)

The model below is the best configuration found by the repository's script-first autoresearch loop. The loop repeatedly edited a constrained candidate file, evaluated each candidate with a stable validation harness, logged results to `experiments/results.csv`, and promoted the best configuration to `experiments/best_config.json`.

## Notebook Plan

This notebook does four things:

1. Loads the Kaggle competition data.
2. Recreates the best autoresearch experiment as a self-contained LightGBM pipeline.
3. Runs 5-fold cross-validation on the training set and generates `submission.csv`.
4. Reflects on the experiment history from `results.csv` and summarizes what worked, what did not, and what should change next time.

## Imports and Configuration

The best experiment used LightGBM because it handled the mixed tabular data well and supported native categorical features. The random seed is fixed for reproducibility.

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

try:
    from lightgbm import LGBMClassifier
except ImportError as exc:
    raise ImportError(
        'This notebook requires LightGBM. On Kaggle, enable internet or add LightGBM to the environment if it is unavailable.'
    ) from exc

SEED = 42
N_SPLITS = 5
TARGET = 'Irrigation_Need'
LABEL_ORDER = ['High', 'Low', 'Medium']


## Data Loading

The cell supports both Kaggle and local repository paths. Kaggle notebooks normally expose the competition files under `/kaggle/input/playground-series-s6e4/`; this repository stores the same files under `data/` for local experimentation.

In [ ]:
def resolve_data_dir():
    candidates = [
        Path('/kaggle/input/playground-series-s6e4'),
        Path('data'),
        Path('/mnt/data'),
    ]
    for path in candidates:
        if (path / 'train.csv').exists() and (path / 'test.csv').exists():
            return path
    raise FileNotFoundError('Could not find train.csv and test.csv in the expected data directories.')

DATA_DIR = resolve_data_dir()
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Data directory: {DATA_DIR}')
print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
train.head()


## Basic Data Check

Before modeling, this quick check verifies the target classes, the submission format, and the mix of numeric and categorical columns. The model code below does not use the Kaggle test labels or public leaderboard feedback.

In [ ]:
target_counts = train[TARGET].value_counts().rename_axis(TARGET).to_frame('count')
target_counts['share'] = target_counts['count'] / len(train)

feature_columns = [col for col in train.columns if col not in ['id', TARGET]]
categorical_columns = train[feature_columns].select_dtypes(include=['object', 'category']).columns.tolist()
numeric_columns = [col for col in feature_columns if col not in categorical_columns]

print(f'Numeric feature count: {len(numeric_columns)}')
print(f'Categorical feature count: {len(categorical_columns)}')
display(target_counts)
display(sample_submission.head())


## Best Autoresearch Experiment

The best logged experiment was `lgbm_tiny_custom_weight_leaves15`. It continued a successful trend found in `results.csv`: conservative LightGBM trees outperformed larger trees, and custom class weighting improved balanced accuracy for the imbalanced target.

Best logged validation results:

| Field | Value |
|---|---:|
| Dev CV balanced accuracy | 0.971166 |
| Internal holdout balanced accuracy | 0.973493 |
| Model | LightGBM multiclass classifier |
| Class weights | `High=18.0`, `Low=0.62`, `Medium=0.83` |
| `num_leaves` | 15 |
| Seed | 42 |

In [ ]:
BEST_EXPERIMENT = {
    'name': 'lgbm_tiny_custom_weight_leaves15',
    'notes': 'Refine leaf-count search around the leaves14 best by testing num_leaves=15.',
    'dev_cv_balanced_accuracy': 0.9711660855905494,
    'holdout_balanced_accuracy': 0.9734934704692769,
    'class_weight': {'High': 18.0, 'Low': 0.62, 'Medium': 0.83},
    'num_leaves': 15,
    'seed': SEED,
}
BEST_EXPERIMENT


## Feature Preparation

The winning candidate intentionally kept feature engineering minimal. It drops only `id` and the target, preserves all original predictors, and converts categorical columns to `pandas.Categorical` with categories aligned across train and test so LightGBM receives consistent category codes.

In [ ]:
def build_features(train_df, test_df):
    y = train_df[TARGET].astype(str)
    X = train_df.drop(columns=[TARGET, 'id'])
    X_test = test_df.drop(columns=['id'])

    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in categorical_cols:
        categories = pd.Index(pd.concat([X[col], X_test[col]], ignore_index=True).astype(str).unique())
        X[col] = pd.Categorical(X[col].astype(str), categories=categories)
        X_test[col] = pd.Categorical(X_test[col].astype(str), categories=categories)

    return X, X_test, y, categorical_cols

X, X_test, y, categorical_cols = build_features(train, test)
print(f'Prepared train matrix: {X.shape}')
print(f'Prepared test matrix: {X_test.shape}')
print(f'Categorical columns passed to LightGBM: {categorical_cols}')


## Model Definition

The model is deliberately small for a gradient boosting tree model. Earlier experiments showed that progressively reducing tree complexity improved balanced accuracy, likely because the original data had strong signal and the larger trees added variance. The custom class weights put more pressure on the rare `High` class, which directly aligns with balanced accuracy.

In [ ]:
def make_model(fold):
    return LGBMClassifier(
        objective='multiclass',
        num_class=len(LABEL_ORDER),
        class_weight=BEST_EXPERIMENT['class_weight'],
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=BEST_EXPERIMENT['num_leaves'],
        max_depth=-1,
        min_child_samples=300,
        subsample=0.9,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_alpha=0.05,
        reg_lambda=0.2,
        random_state=SEED + fold,
        n_jobs=-1,
        verbosity=-1,
    )

def majority_vote(predictions, class_order):
    stacked = np.vstack(predictions).T
    class_rank = {label: idx for idx, label in enumerate(class_order)}
    output = []
    for row in stacked:
        counts = Counter(row)
        output.append(max(counts, key=lambda label: (counts[label], -class_rank[label])))
    return np.asarray(output, dtype=object)


## Cross-Validation and Submission Generation

This cell trains five stratified folds on the full training data, reports out-of-fold balanced accuracy, averages the fold decisions through majority vote, and writes `submission.csv`. The Kaggle test set is used only for final prediction generation.

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_pred = np.empty(len(X), dtype=object)
test_fold_predictions = []
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    model = make_model(fold - 1)
    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx],
        categorical_feature=categorical_cols,
    )

    valid_pred = model.predict(X.iloc[valid_idx])
    oof_pred[valid_idx] = valid_pred
    test_fold_predictions.append(model.predict(X_test))

    fold_score = balanced_accuracy_score(y.iloc[valid_idx], valid_pred)
    fold_scores.append(fold_score)
    print(f'Fold {fold}: balanced_accuracy={fold_score:.10f}')

cv_score = balanced_accuracy_score(y, oof_pred)
test_pred = majority_vote(test_fold_predictions, LABEL_ORDER)

submission = sample_submission.copy()
submission[TARGET] = test_pred
submission.to_csv('submission.csv', index=False)

print(f'OOF balanced accuracy: {cv_score:.10f}')
print('Wrote submission.csv')
display(submission.head())
display(submission[TARGET].value_counts().rename_axis(TARGET).to_frame('predicted_count'))


## Self Reflection from `results.csv`

The repository's `experiments/results.csv` recorded 42 full experiments. The search started from a strong LightGBM baseline and then made small, controlled changes. The clearest improvement path was not adding many engineered features, but instead reducing model complexity and tuning class weights for balanced accuracy.

Key milestones from `results.csv`:

| Experiment | Dev CV balanced accuracy | Holdout balanced accuracy | What it showed |
|---|---:|---:|---|
| `baseline_lgbm_or_histgb` | 0.960633 | 0.963871 | LightGBM was already a strong baseline. |
| `lgbm_balanced_class_weight` | 0.964561 | 0.967450 | Balanced class weighting helped the target metric. |
| `lgbm_balanced_tiny_lower_lr` | 0.970356 | 0.972618 | Tiny trees plus smoother boosting improved generalization. |
| `lgbm_tiny_custom_high_weight_18` | 0.970744 | 0.973437 | Extra `High` class weight helped the rare class. |
| `lgbm_tiny_custom_weight_leaves15` | 0.971166 | 0.973493 | The best dev CV score came from custom class weights and `num_leaves=15`. |

The most important lesson is that the winning direction was conservative. Larger trees, extra feature interactions, and compact feature subsets generally did not beat the simpler all-feature LightGBM setup. The experiment log also shows that holdout accuracy was not perfectly monotonic with CV accuracy, so keeping a holdout guardrail was useful for detecting changes that looked good on CV but were less convincing out-of-sample.

In [ ]:
results_summary = pd.DataFrame([
    {'experiment': 'baseline_lgbm_or_histgb', 'dev_cv': 0.9606334176, 'holdout': 0.9638706794},
    {'experiment': 'lgbm_balanced_class_weight', 'dev_cv': 0.9645605647, 'holdout': 0.9674499689},
    {'experiment': 'lgbm_balanced_tiny_lower_lr', 'dev_cv': 0.9703560527, 'holdout': 0.9726184243},
    {'experiment': 'lgbm_tiny_custom_high_weight_18', 'dev_cv': 0.9707440035, 'holdout': 0.9734366478},
    {'experiment': 'lgbm_tiny_custom_weight_leaves15', 'dev_cv': 0.9711660856, 'holdout': 0.9734934705},
])
results_summary['dev_cv_gain_from_baseline'] = results_summary['dev_cv'] - results_summary.loc[0, 'dev_cv']
results_summary['holdout_gain_from_baseline'] = results_summary['holdout'] - results_summary.loc[0, 'holdout']
display(results_summary)

best_row = results_summary.iloc[-1]
print(f"Best dev CV gain over baseline: {best_row['dev_cv_gain_from_baseline']:.6f}")
print(f"Best holdout gain over baseline: {best_row['holdout_gain_from_baseline']:.6f}")


## Conclusion

This autoresearch run was a success. The best experiment improved dev CV balanced accuracy from `0.960633` to `0.971166`, and improved the internal holdout guardrail from `0.963871` to `0.973493`. The final model is also simple enough to explain and portable enough to run inside a Kaggle notebook.

The main finding is that this competition rewarded disciplined tuning more than broad feature invention. Small LightGBM trees, careful regularization, and custom class weights were more effective than increasing model complexity.

For next time, I would change the research process in these ways:

1. Spawn different workspaces for different ML methods, such as LightGBM, XGBoost, CatBoost, linear models, neural networks, and calibrated ensembles, then compare each method at the end under the same evaluator.
2. Track per-class recall for every run, not just balanced accuracy, so class-weight changes can be understood more directly.
3. Add a small ensemble phase after the single-model search, while keeping the internal holdout as a guardrail against overfitting.
4. Reserve time for parameter stability checks across multiple random seeds before choosing the final submission.

Overall, the submission is a strong, reproducible result from a controlled autoresearch workflow, but the next iteration should compare multiple model families in parallel rather than refining only one successful LightGBM path.